# Assignment 1 — SFT Baseline (TRL + LoRA)

This notebook trains a **Supervised Fine-Tuning (SFT)** baseline model using **Hugging Face TRL** and **PEFT (LoRA)**.

What you get:
- A reproducible SFT run (small model, LoRA adapters)
- Saved model/adapters to disk
- A small fixed-prompt qualitative evaluation saved to `results/baseline_samples.json`

Docs:
- TRL main docs: https://huggingface.co/docs/trl/en/index
- SFTTrainer docs: https://huggingface.co/docs/trl/en/sft_trainer

> Recommended small base model: `Qwen/Qwen2.5-0.5B` (or swap to another small base model).


## Step 1: Install packages

Run this only if your environment is new. It installs the libraries for training.


In [ ]:
# If you're on Colab / a fresh environment, install dependencies:
# (Restart runtime after install if needed)
!pip -q install "transformers>=4.45.0" "datasets>=2.20.0" "accelerate>=0.33.0" \
               "trl>=0.12.0" "peft>=0.12.0" "bitsandbytes>=0.43.0" "sentencepiece" "wandb" "evaluate"

## Step 2: Import and set basic options

This part imports the tools, sets the random seed, makes output folders, and chooses the model and dataset.


In [ ]:
import os, json, random
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# -----------------------
# Reproducibility
# -----------------------
SEED = 42
set_seed(SEED)
random.seed(SEED)

# -----------------------
# Paths
# -----------------------
OUTPUT_DIR = "outputs/sft_baseline_lora"
RESULTS_DIR = "results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# -----------------------
# Model choice (small + base model recommended)
# -----------------------
# Base (non-instruct) model is ideal if you want to demonstrate the SFT step clearly.
MODEL_NAME = "Qwen/Qwen2.5-0.5B"  # base
# MODEL_NAME = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"  # alternative

# -----------------------
# Dataset choice (SFT instruction data)
# -----------------------
SFT_DATASET = "tatsu-lab/alpaca"

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import trl
print(trl.__version__)

0.29.0


## Step 3: Load and format the data

Here we load the Alpaca dataset, keep a smaller sample, turn each example into one text, and split the data into train and test parts.


In [ ]:
# -----------------------
# Load & prepare SFT dataset
# -----------------------
ds = load_dataset(SFT_DATASET, split="train")

# Keep a smaller subset if you're resource-limited (recommended for a first successful run).
MAX_TRAIN_SAMPLES = 10_000
if len(ds) > MAX_TRAIN_SAMPLES:
    ds = ds.shuffle(seed=SEED).select(range(MAX_TRAIN_SAMPLES))

def format_alpaca(example):
    """Convert Alpaca schema -> a single training text.

    Alpaca fields: instruction, input, output.
    We combine them into one text sequence for causal LM training.
    """
    instr = (example.get("instruction") or "").strip()
    inp = (example.get("input") or "").strip()
    out = (example.get("output") or "").strip()

    if inp:
        prompt = f"### Instruction:\n{instr}\n\n### Input:\n{inp}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instr}\n\n### Response:\n"
    return {"text": prompt + out}

ds = ds.map(format_alpaca, remove_columns=ds.column_names)

# Train/validation split (for basic monitoring)
split = ds.train_test_split(test_size=0.02, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]

train_ds, eval_ds

(Dataset({
     features: ['text'],
     num_rows: 9800
 }),
 Dataset({
     features: ['text'],
     num_rows: 200
 }))

## Step 4: Load tokenizer and base model

This step loads the tokenizer and model. It also sets the pad token and uses memory-saving options.


In [ ]:
# -----------------------
# Tokenizer / Model
# -----------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# Some base models don't have a pad token; use eos as pad to avoid Trainer issues.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

# Enable gradient checkpointing to reduce memory (optional; helps on smaller GPUs)
model.gradient_checkpointing_enable()
model.config.use_cache = False

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:01<00:00, 226.96it/s, Materializing param=model.norm.weight]                              


## Step 5: Add LoRA

LoRA trains a small set of extra weights. This makes fine-tuning cheaper and faster.


In [ ]:
# -----------------------
# LoRA config (PEFT)
# -----------------------
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Works for many transformer LMs; if your model errors, inspect module names and adjust.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

## Step 6: Set training options

This part sets the training arguments and builds the SFT trainer.


In [ ]:
# -----------------------
# SFT training config (TRL)
# -----------------------
# If you have limited VRAM, reduce:
# - per_device_train_batch_size
# - max_seq_length
# and/or increase gradient_accumulation_steps.
sft_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    max_length=512,
    packing=False,
    bf16=torch.cuda.is_available(),
    fp16=False,
    report_to=[],  # change to [] if you don't want wandb
    run_name="sft_baseline_lora",
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## Step 7: Train the model

Now the trainer learns from the SFT data.


In [ ]:
# -----------------------
# Train
# -----------------------
train_result = trainer.train()
train_result

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/opt/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss


RuntimeError: MPS backend out of memory (MPS allocated: 15.02 GiB, other allocations: 1.94 GiB, max allowed: 18.13 GiB). Tried to allocate 1.90 GiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

## Step 8: Save the result

This saves the LoRA adapter and the tokenizer to disk.


In [ ]:
# -----------------------
# Save adapters + tokenizer
# -----------------------
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to:", OUTPUT_DIR)

## Step 9: Test with a few prompts

We ask the model a few simple questions and save the answers. This gives a quick baseline check.


In [ ]:
# -----------------------
# Qualitative baseline samples (fixed prompts)
# -----------------------
# These are useful for your report & slides.
prompts = [
    "Explain the difference between supervised learning and reinforcement learning in simple terms.",
    "Write a short email to a tutor asking for an extension, politely.",
    "Given a list of numbers [3, 1, 4, 1, 5], what is the mean and median?",
    "Summarize the pros and cons of using PPO vs DPO for RLHF.",
]

def generate(model, tokenizer, prompt, max_new_tokens=160):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

samples = [{"prompt": p, "output": generate(trainer.model, tokenizer, p)} for p in prompts]

out_path = os.path.join(RESULTS_DIR, "baseline_samples.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(samples, f, ensure_ascii=False, indent=2)

out_path, samples[0]["output"][:300]

## Next steps (for your assignment)

1) **Reward Modeling (RM)**: convert preference datasets into `prompt, chosen, rejected` and train a reward model.  
2) **RLHF**: run DPO (easiest) or PPO using TRL.  
3) **Evaluation**: keep a fixed prompt set, compute win-rate with held-out preference pairs, and do blind human comparisons (baseline vs RLHF).
